In [2]:
import os
import sys
print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\research


In [3]:
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\src")
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01")

In [4]:
%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Regression_01\\research'

In [5]:
os.chdir("../")

In [6]:
%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Regression_01'

In [7]:
import box
print(box.__version__)

7.4.1


In [8]:
# entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:

    root_dir: Path
    model_report_path: Path
    trained_model_path: Path
    evaluation_report_path: Path
    threshold: float
    metric_name: str
    save_metrics: bool


In [9]:
from Regression_01.entity.config_entity import ModelEvaluationConfig
from Regression_01.utils.common import read_yaml, create_directories
from Regression_01.constant import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from Regression_01.config.configuration import ConfigurationManager

In [10]:
# configuration manager

def get_model_evaluation_config(self) -> ModelEvaluationConfig:

    config = self.config.model_evaluation

    params = self.params.model_evaluation

    create_directories([config.root_dir])

    model_evaluation_config = ModelEvaluationConfig(

        root_dir=Path(config.root_dir),

        model_report_path=Path(config.model_report_path),

        trained_model_path=Path(config.trained_model_path),

        evaluation_report_path=Path(config.evaluation_report_path),

        threshold=params.threshold,

        metric_name=params.metric,

        save_metrics=params.save_metrics
    )

    return model_evaluation_config

In [11]:
import pandas as pd

from Regression_01.logging import logger
from Regression_01.entity.config_entity import ModelEvaluationConfig

In [12]:
# components


class ModelEvaluation:

    def __init__(
        self,
        config: ModelEvaluationConfig
    ):

        self.config = config


    def initiate_model_evaluation(self):

        # STEP 1 : Load Model Comparison Report

        logger.info("Loading model comparison report")

        comparison_df = pd.read_csv(
            self.config.model_report_path
        )

        # STEP 2 : Get Best Model

        logger.info("Finding best model")

        best_model = comparison_df.iloc[0]

        best_model_name = best_model["Model"]

        best_r2_score = best_model["R2 Score"]


        # STEP 3 : Read Threshold

        logger.info("Reading evaluation threshold")

        threshold = self.config.threshold


        # STEP 4 : Compare Best Model with Threshold

        logger.info("Evaluating model")

        if best_r2_score >= threshold:

            status = "Accepted"

        else:

            status = "Rejected"


        # STEP 5 : Create Evaluation Report

        logger.info("Creating evaluation report")

        evaluation_df = pd.DataFrame({

            "Best Model":[best_model_name],

            "R2 Score":[best_r2_score],

            "Threshold":[threshold],

            "Status":[status]

        })

        # STEP 6 : Save Evaluation Report

        if self.config.save_metrics:

            logger.info("Saving evaluation report")

            evaluation_df.to_csv(

                self.config.evaluation_report_path,

                index=False

            )

            logger.info("Evaluation report saved successfully")

        # STEP 7 : Print Final Status

        logger.info(f"Best Model : {best_model_name}")

        logger.info(f"R2 Score : {best_r2_score:.4f}")

        logger.info(f"Status : {status}")


        return evaluation_df

In [13]:
# Pipeline

STAGE_NAME = "Model Evaluation Stage"

class ModelEvaluationTrainingPipeline:

    def __init__(self):
        pass

    def main(self):

        config = ConfigurationManager()

        model_evaluation_config = config.get_model_evaluation_config()

        model_evaluation = ModelEvaluation(
            config=model_evaluation_config
        )

        model_evaluation.initiate_model_evaluation()

In [14]:
obj = ModelEvaluationTrainingPipeline()

obj.main()

[2026-08-04 00:54:05,222: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\config\config.yaml loaded successfully]
[2026-08-04 00:54:05,228: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\params.yaml loaded successfully]
[2026-08-04 00:54:05,238: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\schema.yaml loaded successfully]
[2026-08-04 00:54:05,238: INFO: common: created directory at artifacts]
[2026-08-04 00:54:05,238: INFO: common: created directory at artifacts/model_evaluation]
[2026-08-04 00:54:05,243: INFO: 2701599146: Loading model comparison report]
[2026-08-04 00:54:05,267: INFO: 2701599146: Finding best model]
[2026-08-04 00:54:05,275: INFO: 2701599146: Reading evaluation threshold]
[2026-08-04 00:54:05,275: INFO: 2701599146: Evaluating model]
[2026-08-04 00:54:05,280: INFO: 2701599146: Creating evaluation report]
[2026-08-04 00:54:05,283: INFO: 2701599146: Saving 

[2026-08-04 00:54:05,296: INFO: 2701599146: Best Model : XGBoost]
[2026-08-04 00:54:05,298: INFO: 2701599146: R2 Score : 0.6589]
[2026-08-04 00:54:05,299: INFO: 2701599146: Status : Accepted]
